In [2]:
import torch
from torch import nn
import numpy as np

with open('data/input.txt', 'r') as f:
    text = f.read()
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [3]:
chars

['\n',
 ' ',
 '!',
 '$',
 '&',
 "'",
 ',',
 '-',
 '.',
 '3',
 ':',
 ';',
 '?',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [4]:
vocab_size

65

In [5]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: string to list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: list of integers to string
data = torch.tensor(encode(text))

In [6]:
n = int(0.9 * len(data))
train_data = data[:n].to("cuda")
val_data = data[n:].to("cuda")

block_size = 8 # context length: how many characters do we look at?
batch_size = 4 # how many independent sequences will we process in parallel?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data_set = train_data if split == 'train' else val_data

    # Pick random starting points in the text
    # We subtract block_size so we don't go out of bounds at the very end
    ix = torch.randint(len(data_set) - block_size, (batch_size,))

    # Stack the chunks into a matrix (Batch Size x Block Size)
    x = torch.stack([data_set[i:i+block_size] for i in ix])
    y = torch.stack([data_set[i+1:i+block_size+1] for i in ix])

    return x, y

# Let's see it in action
xb, yb = get_batch('train')
print('inputs (xb):')
print(xb.shape)
print(xb)
print('targets (yb):')
print(yb.shape)
print(yb)

inputs (xb):
torch.Size([4, 8])
tensor([[60, 43,  1, 50, 53, 52, 45,  1],
        [50, 53, 53, 42,  6,  1, 40, 39],
        [53, 58, 46, 43, 56,  5, 57,  1],
        [56, 56, 43, 50,  1, 53, 44,  1]], device='cuda:0')
targets (yb):
torch.Size([4, 8])
tensor([[43,  1, 50, 53, 52, 45,  1, 39],
        [53, 53, 42,  6,  1, 40, 39, 58],
        [58, 46, 43, 56,  5, 57,  1, 39],
        [56, 43, 50,  1, 53, 44,  1, 58]], device='cuda:0')


In [7]:
n_embd = 128 # Every character is now a vector of 32 numbers
head_size = 128
from torch.nn import functional as F
class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)

        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # hide the future!
        wei = F.softmax(wei, dim=-1) # (B, T, T)

        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, hs)
        return out
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )
        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        # run all heads in parallel
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # mix their outputs
        out = self.proj(out)
        return out
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)

        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU(),
        )
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # talk to others
        x = x + self.ffwd(self.ln2(x)) # think alone
        return x

class NanoGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, head_size):
        super().__init__()
        # 1. Map character IDs to "thought vectors" (embeddings)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # 2. Position is important! (The model needs to know where it is in the sentence)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # 3. The "Communication" layer (Self-Attention)
        n_head = 16
        head_size = n_embd // n_head
        self.sa = nn.ModuleList([Block(n_embd, n_head) for _ in range(n_head)])




        # 4. The "Computation" layer (Think about what you just heard)
        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU(),
        )

        # 5. The final output layer (Map back to vocab size)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C) - Combine what it is with WHERE it is

        for block in self.sa:
            x = block(x)

   # Apply feed-forward (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
model = NanoGPT(vocab_size, n_embd, head_size).to("cuda")
# Run your training loop again! peak cinema dfkjkfjdkfdjkfkfjdfkjdkfjkdjfkdjfkdjkfjkfddfjkfkfkdjkfdjkfkjdfjkdjfkdjfdkjfdkjfdkdfjdjkdjfkdjdkfjdkjfkdkfjdfk

In [8]:
@torch.no_grad()
def generate(model, start_seq, max_new_tokens=100):
    model.eval()
    device = next(model.parameters()).device
    idx = torch.tensor([start_seq], device=device)

    for _ in range(max_new_tokens):
        # 👇 THIS IS THE IMPORTANT LINE
        idx_cond = idx[:, -block_size:]

        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]      # last time step
        probs = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)

        idx = torch.cat([idx, next_idx], dim=1)

    return idx[0].tolist()


In [9]:

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for iter in range(5000): # increase this to 5000+ for better results
    xb, yb = get_batch('train')# You'll need a get_batch function to feed it data
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 100 == 0:
        print(f"step {iter}: loss {loss.item():.4f}")

step 0: loss 6.6916


KeyboardInterrupt: 

In [10]:
tnr = torch.tensor([[1, 2, 3, 4, 5, 6],
                    [7, 8, 9, 10, 11, 12]])
print(tnr.transpose(0, 1))

tensor([[ 1,  7],
        [ 2,  8],
        [ 3,  9],
        [ 4, 10],
        [ 5, 11],
        [ 6, 12]])


In [11]:
# convert characters to integers
start_text = "O"  # seed
start_seq = [stoi[c] for c in start_text]

generated = generate(model, start_seq, max_new_tokens=200)
print("".join([itos[i] for i in generated]))


Oqd wi thir sdNe ousiicoCor sr t
reKiregI;
s :o.AuEinddi t win Wii:y gouriur str wCil  uhi  urouirEr y houN!sh LsnonouiEor wicye;uels feage !
NII
lknI s y Drhe ?
brekLs
LdI; winevedeinourevr w

r st It


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 7.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 6.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 7.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "distilbert/distilgpt2"

# This forces the download of the .safetensors version
model = AutoModelForCausalLM.from_pretrained(model_id, use_safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from safetensors import safe_open

with safe_open("model.safetensors", framework="pt") as f:
    print(list(f.keys())[:5])


FileNotFoundError: No such file or directory: model.safetensors

In [ ]:
!ls

acc_results.csv      models	  torch1.ipynb	torch5.ipynb
data		     __pycache__  torch2.ipynb	torch6.ipynb
helper_functions.py  pyg.py	  torch3.ipynb	torch7.ipynb
model_functions.py   runs	  torch4.ipynb	torch8.ipynb


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "microsoft/phi-3-mini-4k-instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb
)


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

prompt = "explain black holes like im 6"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,     # creativity knob
        top_p=0.9,           # dont let it go insane
        do_sample=True
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


explain black holes like im 6th grader. I'm trying to explain it to my little brother and I'm struggling. Can you simplify this for me? ----- Dear Alice, Black holes are very fascinating objects in space. Imagine space as a huge sheet of paper. Now, picture a heavy ball in the middle of this sheet. The ball will make a big dip or a 'well', right? This is kind of like what a black hole does. It's a place in space where the gravity is so strong, it's like the 'well' is much deeper. It's so deep that even light, which is the fastest thing in the universe, can't escape from it once it gets too close. That's why we call it a 'black' hole. But don't worry, they're far away from us and won't harm us. They're just like the heavy ball on our paper. They're just space


In [1]:
from datasets import Dataset


dataset = Dataset.from_text("data/input.txt")


In [7]:
#model = AutoModelForCausalLM.from_pretrained(
#    model_id,
#    quantization_config=bnb,
#    device_map="auto"
#)

from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)

from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)


In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./finetune",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=500,
    fp16=True,
    logging_steps=10,
    bf16 = False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

trainer.train()


/home/sam/ml_worl/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [24]:
for name, module in model.named_modules():
    print(name)



model
model.embed_tokens
model.layers
model.layers.0
model.layers.0.self_attn
model.layers.0.self_attn.o_proj
model.layers.0.self_attn.qkv_proj
model.layers.0.mlp
model.layers.0.mlp.gate_up_proj
model.layers.0.mlp.down_proj
model.layers.0.mlp.activation_fn
model.layers.0.input_layernorm
model.layers.0.post_attention_layernorm
model.layers.0.resid_attn_dropout
model.layers.0.resid_mlp_dropout
model.layers.1
model.layers.1.self_attn
model.layers.1.self_attn.o_proj
model.layers.1.self_attn.qkv_proj
model.layers.1.mlp
model.layers.1.mlp.gate_up_proj
model.layers.1.mlp.down_proj
model.layers.1.mlp.activation_fn
model.layers.1.input_layernorm
model.layers.1.post_attention_layernorm
model.layers.1.resid_attn_dropout
model.layers.1.resid_mlp_dropout
model.layers.2
model.layers.2.self_attn
model.layers.2.self_attn.o_proj
model.layers.2.self_attn.qkv_proj
model.layers.2.mlp
model.layers.2.mlp.gate_up_proj
model.layers.2.mlp.down_proj
model.layers.2.mlp.activation_fn
model.layers.2.input_layerno

In [26]:
torch.cuda.empty_cache()